In [1]:
import pandas as pd
import numpy as np
from src.data.metrics import get_metrics, exclude_tuning_samples
from src.data.utils import load_data, get_trajectories, get_discretization

In [2]:
N_TUNE_800 = 8839
N_TUNE_400 = 5000

In [3]:
dfo = pd.read_json("../reports/online_v_batch.jsonl", lines=True, orient="records")
dfo["eval_mode"] = "Online"
metrics = ["top_1_accuracy", "top_5_accuracy", "macro_f1"]
get_metrics(dfo, metrics)
dfb = pd.read_json("../reports/batch_20_full_v2.jsonl", lines=True, orient="records")
dfb["eval_mode"] = "Batch"
df = pd.concat([dfo, dfb])

df["Embedding"] = df["loc_levels"].apply(lambda x: "MiLES" if x == 5 else "Default")
dfm = df.groupby(["dataset", "Embedding", "eval_mode"])[metrics].mean()

df_delta = dfm.xs("MiLES", level=1) / dfm.xs("Default", level=1) - 1
df_table = df_delta.pivot_table(index="eval_mode", columns="dataset", sort=False)
df_table = df_table.sort_index(axis=1, level=1)
df_table.columns = df_table.columns.reorder_levels([1, 0])

df_table = df_table.reindex(columns=metrics, level=1)
df_table = df_table.rename(
    columns={
        "foursquare_NYC": "Foursquare-NYC",
        "foursquare_TKY": "Foursquare-TKY",
        "geolife": "GeoLife",
    }
)
df_table = df_table.rename(
    columns={
        "top_1_accuracy": "Acc@1",
        "top_5_accuracy": "Acc@5",
        "macro_f1": "Macro F1",
    }
)
print((df_table * 100).style.background_gradient(
        cmap="RdYlGn", vmin=-30, vmax=30, axis=None, text_color_threshold=0.2
    ).format("{:+.2f}").to_latex(hrules=True, convert_css=True))

\begin{tabular}{lrrrrrrrrr}
\toprule
dataset & \multicolumn{3}{r}{Foursquare-NYC} & \multicolumn{3}{r}{Foursquare-TKY} & \multicolumn{3}{r}{GeoLife} \\
 & Acc@1 & Acc@5 & Macro F1 & Acc@1 & Acc@5 & Macro F1 & Acc@1 & Acc@5 & Macro F1 \\
eval_mode &  &  &  &  &  &  &  &  &  \\
\midrule
Batch & {\cellcolor[HTML]{E3F399}} \color[HTML]{000000} +4.24 & {\cellcolor[HTML]{D5ED88}} \color[HTML]{000000} +6.41 & {\cellcolor[HTML]{E6F59D}} \color[HTML]{000000} +3.84 & {\cellcolor[HTML]{E6F59D}} \color[HTML]{000000} +3.97 & {\cellcolor[HTML]{E0F295}} \color[HTML]{000000} +4.80 & {\cellcolor[HTML]{E6F59D}} \color[HTML]{000000} +3.93 & {\cellcolor[HTML]{E0F295}} \color[HTML]{000000} +4.81 & {\cellcolor[HTML]{EFF8AA}} \color[HTML]{000000} +2.52 & {\cellcolor[HTML]{C7E77F}} \color[HTML]{000000} +8.10 \\
Online & {\cellcolor[HTML]{E3F399}} \color[HTML]{000000} +4.32 & {\cellcolor[HTML]{C7E77F}} \color[HTML]{000000} +8.04 & {\cellcolor[HTML]{DDF191}} \color[HTML]{000000} +5.31 & {\cellcolor[HTML]{E3F399